In [ ]:
import pandas as pd
import requests
import concurrent.futures
from tqdm import tqdm

In [6]:
# fetch metadata

def fetch_metadata(id, metadata_path):
    """
    Fetch metadata for an entry from InterPro.

    Args:
        id (str): ID (e.g., "PIRSF500444")
        metadata_path (str): Path to save the fetched metadata (e.g. pirsf)

    Returns:
        dict: Metadata dictionary containing details about the entry.
    """
    url = f"https://www.ebi.ac.uk/interpro/api/entry/{metadata_path}/{id}/"
    headers = {"Accept": "application/json"}
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Error fetching {metadata_path} data for {id}: HTTP {response.status_code}")
        return {}
    
    data = response.json()
    return data

In [ ]:
# open up swissprot_filtered_with_alphafold_link.tsv.gz

df = pd.read_csv("swissprot_filtered_with_alphafold_link.tsv.gz", sep="\t")

# print the first 5 rows
print(df.head())

# print the last 5 rows
print(df.tail())

/var/folders/r9/b7wl1nls18g445wkmxrxdgqm0000gn/T/ipykernel_13486/4279429553.py:3: DtypeWarning: Columns (12,14,19,37,38,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("swissprot_filtered_with_alphafold_link.tsv.gz", sep="\t")


   Unnamed: 0       Entry   Entry Name             Gene Names  \
0           0  A0A009IHW8  ABTIR_ACIB9              J512_3302   
1           1  A0A023I7E1   ENG1_RHIMI            ENG1 LAM81A   
2           2  A0A024RXP8   GUX1_HYPJR  cbh1 M419DRAFT_125125   
3           3  A0A024SC78  CUTI1_HYPJR        M419DRAFT_76732   
4           4  A0A024SH76   GUX2_HYPJR  cbh2 M419DRAFT_122470   

                                            Organism  \
0           Acinetobacter baumannii (strain 1295743)   
1                                  Rhizomucor miehei   
2  Hypocrea jecorina (strain ATCC 56765 / BCRC 32...   
3  Hypocrea jecorina (strain ATCC 56765 / BCRC 32...   
4  Hypocrea jecorina (strain ATCC 56765 / BCRC 32...   

                                       Protein names  Length   Mass  \
0  2' cyclic ADP-D-ribose synthase AbTIR (2'cADPR...     269  30922   
1  Glucan endo-1,3-beta-D-glucosidase 1 (Endo-1,3...     796  89495   
2  Exoglucanase 1 (EC 3.2.1.91) (1,4-beta-cellobi...     51

In [ ]:
# test fetch metadata wit pfam
df_pfam = df[df["Pfam"].notna()]
df_pfam["Pfam"].iloc[0]

,Unnamed: 0,Entry,Entry Name,Gene Names,Organism,Protein names,Length,Mass,Sequence,Active site,...,PDB,AlphaFoldDB,CDD,HAMAP,PANTHER,PIRSF,PRINTS,SUPFAM,Gene3D,alphafold_link
0,0,A0A009IHW8,ABTIR_ACIB9,J512_3302,Acinetobacter baumannii (strain 1295743),2' cyclic ADP-D-ribose synthase AbTIR (2'cADPR...,269,30922,MSLEQKKGADIISKILQIQNSIGKTTSPSTLKTKLSEISRKEQENA...,"ACT_SITE 208; /evidence=""ECO:0000255|PROSITE-P...",...,7UWG;7UXU;8G83;,A0A009IHW8;,NaN,NaN,NaN,NaN,NaN,SSF52200;,3.40.50.10140;,https://alphafold.ebi.ac.uk/api/prediction/A0A...
1,1,A0A023I7E1,ENG1_RHIMI,ENG1 LAM81A,Rhizomucor miehei,"Glucan endo-1,3-beta-D-glucosidase 1 (Endo-1,3...",796,89495,MRFQVIVAAATITMITSYIPGVASQSTSDGDDLFVPVSNFDPKSIF...,"ACT_SITE 500; /evidence=""ECO:0000255|PROSITE-P...",...,4K35;4K3A;5XBZ;5XC2;,A0A023I7E1;,NaN,NaN,PTHR31983;PTHR31983:SF0;,NaN,NaN,NaN,1.10.287.1170;2.70.98.30;1.20.5.420;,https://alphafold.ebi.ac.uk/api/prediction/A0A...
2,2,A0A024RXP8,GUX1_HYPJR,cbh1 M419DRAFT_125125,Hypocrea jecorina (strain ATCC 56765 / BCRC 32...,"Exoglucanase 1 (EC 3.2.1.91) (1,4-beta-cellobi...",514,54111,MYRKLAVISAFLATARAQSACTLQSETHPPLTWQKCSSGGTCTQQT...,"ACT_SITE 229; /note=""Nucleophile""; /evidence=""...",...,NaN,A0A024RXP8;,cd07999;,NaN,PTHR33753;PTHR33753:SF2;,NaN,PR00734;,SSF57180;SSF49899;,2.70.100.10;,https://alphafold.ebi.ac.uk/api/prediction/A0A...
3,3,A0A024SC78,CUTI1_HYPJR,M419DRAFT_76732,Hypocrea jecorina (strain ATCC 56765 / BCRC 32...,Cutinase (EC 3.1.1.74),248,25924,MRSLAILTTLLAGHAFAYPKPAPQSVNRRDWPSINEFLSELAKVMP...,"ACT_SITE 164; /note=""Nucleophile""; /evidence=""...",...,4PSC;4PSD;4PSE;,A0A024SC78;,NaN,NaN,PTHR48250:SF1;PTHR48250;,NaN,PR00129;,SSF53474;,3.40.50.1820;,https://alphafold.ebi.ac.uk/api/prediction/A0A...
4,4,A0A024SH76,GUX2_HYPJR,cbh2 M419DRAFT_122470,Hypocrea jecorina (strain ATCC 56765 / BCRC 32...,"Exoglucanase 2 (EC 3.2.1.91) (1,4-beta-cellobi...",471,49653,MIVGILTTLATLATLAASVPLEERQACSSVWGQCGGQNWSGPTCCA...,"ACT_SITE 245; /note=""Proton donor""; /evidence=...",...,NaN,A0A024SH76;,NaN,NaN,PTHR34876;PTHR34876:SF4;,PIRSF001100;,PR00733;,SSF57180;SSF51989;,3.20.20.40;,https://alphafold.ebi.ac.uk/api/prediction/A0A...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536646,536646,Q9ZVR1,PP2B5_ARATH,PP2B5 At2g02300 T16F16.9,Arabidopsis thaliana (Mouse-ear cress),F-box protein PP2-B5 (Protein PHLOEM PROTEIN 2...,284,32015,MGQKHGVDTRGKGAEFCGCWEILTEFINGSSASFDDLPDDCLAIIS...,NaN,...,NaN,Q9ZVR1;,cd22162;,NaN,PTHR32278;PTHR32278:SF57;,NaN,NaN,SSF81383;,NaN,https://alphafold.ebi.ac.uk/api/prediction/Q9ZVR1
536647,536647,Q9ZVR3,PP2B4_ARATH,PP2B4 At2g02280 T16F16.7,Arabidopsis thaliana (Mouse-ear cress),Putative protein PHLOEM PROTEIN 2-LIKE B4 (AtP...,144,16551,MNTQILSQKTRYSAYIVYKTIYRFHGFKHIGVGFIGHGTPKAKRWE...,NaN,...,NaN,Q9ZVR3;,NaN,NaN,PTHR32278;PTHR32278:SF57;,NaN,NaN,NaN,NaN,https://alphafold.ebi.ac.uk/api/prediction/Q9ZVR3
536648,536648,Q9ZW38,FBK36_ARATH,At2g29600 F16P2.2,Arabidopsis thaliana (Mouse-ear cress),F-box/kelch-repeat protein At2g29600,415,47492,MASISETSDDGSNGGDPNQKPEEPHKNPQEGKEEENQNEKPKEDDH...,NaN,...,NaN,Q9ZW38;,NaN,NaN,PTHR24414:SF65;PTHR24414;,NaN,NaN,SSF117281;,2.120.10.80;,https://alphafold.ebi.ac.uk/api/prediction/Q9ZW38
536649,536649,Q9ZWC6,ATB_ARATH,ATB At1g55590 F20N2.2,Arabidopsis thaliana (Mouse-ear cress),F-box protein At-B,607,66260,MEEVTRSVLAEEILKRLDLENLCSVACVSTTLRSAVVSGVLPSLTS...,NaN,...,NaN,Q9ZWC6;,NaN,NaN,PTHR13318:SF176;PTHR13318;,NaN,NaN,SSF52047;,3.80.10.10;,https://alphafold.ebi.ac.uk/api/prediction/Q9ZWC6


In [12]:
def process_column_efficiently(
    df, 
    interpro_column='PIRSF', 
    max_workers=10, 
    batch_size=50,
    output_column='pirsf_info',
    metadata_path='pirsf'
):
    """
    Efficiently process InterPro column and fetch information for each entry in batches.
    """
    import concurrent.futures
    from tqdm import tqdm
    import pandas as pd

    # Create copy
    result_df = df.copy()
    result_df[output_column] = [[] for _ in range(len(df))]

    # Get all unique IDs
    all_interpro_ids = set()
    for value in df[interpro_column].dropna():
        if isinstance(value, str):
            ids = [id.strip() for id in value.split(';') if id.strip()]
            all_interpro_ids.update(ids)

    all_interpro_ids = list(all_interpro_ids)
    print(f"Found {len(all_interpro_ids)} unique InterPro IDs to fetch")

    # Cache for fetched results
    interpro_cache = {}

    def fetch_batch(interpro_ids_batch):
        """Fetch multiple InterPro entries in one go"""
        results = {}
        for interpro_id in interpro_ids_batch:
            try:
                data = fetch_metadata(interpro_id, metadata_path)
                if data:
                    results[interpro_id] = data
            except Exception as e:
                print(f"Error fetching {interpro_id}: {e}")
        return results

    # Break IDs into batches
    batches = [all_interpro_ids[i:i+batch_size] for i in range(0, len(all_interpro_ids), batch_size)]

    print(f"Fetching InterPro data in {len(batches)} batches...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_batch, batch): tuple(batch) for batch in batches}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
            batch_results = future.result()
            interpro_cache.update(batch_results)

    print(f"Successfully fetched {len(interpro_cache)} InterPro entries")

    # Populate DataFrame
    print("Processing rows...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        value = row[interpro_column]
        if pd.isna(value) or not isinstance(value, str):
            continue
        interpro_ids = [id.strip() for id in value.split(';') if id.strip()]
        row_interpro_info = [interpro_cache[id] for id in interpro_ids if id in interpro_cache]
        result_df.at[idx, output_column] = row_interpro_info
        
    # save to tsv.gz
    result_df.to_csv(f"swissprot_filtered_with_alphafold_link_and_{metadata_path}.tsv.gz", sep="\t", index=False, compression='gzip')

    return result_df


In [ ]:
# run for Pfam

process_column_efficiently(
    df,
    interpro_column='Pfam',
    output_column='pfam_info',
    metadata_path='pfam'
)

Found 16073 unique InterPro IDs to fetch
Fetching InterPro data in 322 batches...


python(14639) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
  3%|▎         | 10/322 [00:53<10:50,  2.08s/it] 

Error fetching pfam data for PF19257: HTTP 410


 21%|██        | 68/322 [05:40<09:10,  2.17s/it]  

Error fetching pfam data for PF12418: HTTP 410


 49%|████▉     | 159/322 [13:01<04:29,  1.66s/it]